# OBJECTIVE: Feature Engineering and Aggregation Strategy for Road Analysis
This notebook aims to generate a CSV file containing the key characteristics of roads within a specific sector. Starting from historical accident databases, the process involves filtering by sector, then grouping the data by municipality and road. Data aggregation is handled on a case-by-case basis, either by calculating the mode of a column or by applying specific custom functions.

KEY GOALS:
- **Feature Selection & Structure:** Identification and extraction of relevant columns to build the core backbone of the database.
- **Data Profiling & Filtering:** Granular analysis by municipality and road to identify missing values, outliers, and data distribution.
- **Aggregation Logic:** Definition of the mathematical strategy for each variable (Min, Max, Mode, Mean) to condense historical data into fixed characteristics per segment.
- **Master Dataset Generation:** Export of a reference CSV file linking each road in the sector to its physical characteristics and most frequent accident typology.

SECTORS:
- Bassens: 33032
- Sainte-Eulalie: 33397
- Carbon-Blanc: 33096
- Yvrac: 33554
- Ambarès-et-Lagrave: 33003
- Lormont:  33249


In [ ]:
import pandas as pd
import os
from pathlib import Path
import sys
sys.path.append(os.path.abspath(os.path.join('..')))
import src.data_processing.schema_manager as SchemaManager
from src.config import SCHEMA_FILE_PATH
from src.common_utils import read_yaml
from collections import Counter

# Open the CSV file issue from 03_merge processing keeping the 'adr' column when line are merged by usager
df = pd.read_csv("Merge_address.csv", sep=',')
df.shape
df.head()


Filter by municipality code to retain only the records belonging to the fire station's specific sector.

In [ ]:
secteur = {    
    "Bassens": 33032,
    "Sainte-Eulalie": 33397,
    "Carbon-Blanc":33096,
    "Yvrac":33554,
    "Ambarès-et-Lagrave":33003,
    "Lormont": 33249
}

secteur_com_list = [str(code) for code in secteur.values()]

print(secteur_com_list)
df_secteur = df[df['com'].isin(secteur_com_list)]
print(df_secteur.shape)
df_secteur.head(5)

In [ ]:
path_for_notebook = Path(os.path.join("..", SCHEMA_FILE_PATH))
schema_mgr = SchemaManager.SchemaManager(read_yaml(path_for_notebook))

structure_cols = ["catr",
    "v1",
    "circ",
    "vosp",
    "prof",
    "plan",
    "larrout",
    "surf",
    "infra",
    "vma",
    "dep",
    "com",
    "agg",
    "lat",
    "long",
]

date_cols = ["jour","mois","an",
"hrmn",
"week_day",
"Hours",
]

meteo_cols = ["lum","atm",'surf']

accident_a_regarder = [
    "situ",
    "int",
    "col",
    "obs",
    "obsm",
    "locp",
]

accident_cols = [
    "Num_Acc",
    "nbv",
    "id_vehicule",
    "num_veh",
    "senc",
    'choc',
    "manv",
    "motor",
    "place",
    "catv_cluster",
    "grav",
    "sexe",
    "trajet",
    "actp",
    "etatp",
    "age",
    "secu_merged",
    "count_indemne",
    "count_blesse_leger",
    "count_blesse_hosp",
    "count_tue"
]

all_cols = structure_cols + date_cols + accident_a_regarder + accident_cols + meteo_cols

print("\nUndefined columns:")
for col in df_secteur.columns:
    if col not in all_cols:
        print(col)

print("\nUndefined columns:")
for col in df_secteur.columns:
    if col not in all_cols:
        print(col,": ",schema_mgr.get_description(col))

print("\nColumns defined in structure database:")
for col in df_secteur.columns:
    if col in structure_cols:
        print(col,": ",schema_mgr.get_description(col))

print("\nColumns defined with date:")
for col in df_secteur.columns:
    if col in date_cols:
        print(col,": ",schema_mgr.get_description(col))

print("\nWeather-related columns:")
for col in df_secteur.columns:
    if col in meteo_cols:
        print(col,": ",schema_mgr.get_description(col))

print("\nFeatures to review:")
for col in df_secteur.columns:
    if col in accident_a_regarder:
        print(col,": ",schema_mgr.get_description(col))

print("\nOther accident-related columns:")
for col in df_secteur.columns:
    if col in accident_cols:
        print(col,": ",schema_mgr.get_description(col))


**[structure_cols]:** Definition of the mathematical strategy for each variable (Min, Max, Mode, Mean) to condense historical data into fixed characteristics per segment.

In [ ]:
strut_car = df[["adr"]+structure_cols]

strut_car.head()

for commune in secteur_com_list:
    df_com = strut_car[strut_car['com'] == commune]
    for road in df_com['adr'].unique():
        df_road = df_com[df_com['adr'] == road]
        print("-" * 20)
        print(df_road.head(10))

Specifying aggregation functions applied to grouped data.

In [ ]:
def get_mode(series):
    if series.empty: return None
    m = series.mode()
    return m.iloc[0] if not m.empty else None


def get_max_if_significant(series, v_min, v_max, threshold=0.05):
    """
    Filters values between v_min and v_max, then returns the max 
    if it represents more than 'threshold'% of the total data.
    Otherwise, falls back to the global mode.
    """
    if series.empty:
        return None
    
    # 1. Keep only values in the business range (e.g., 1 to 4 for plan)
    in_range = series[series.between(v_min, v_max)]
    
    if in_range.empty:
        return series.mode().iloc[0] if not series.mode().empty else None

    # 2. Calculate frequencies on the total series (to keep the real proportion)
    counts = series.value_counts(normalize=True)
    
    # 3. Find the highest value in our range
    highest_in_range = in_range.max()
    
    # 4. If this high value is significant (> 5%), take it
    if counts.get(highest_in_range, 0) >= threshold:
        return highest_in_range
    
    # 5. Otherwise, return the mode (most frequent value)
    return series.mode().iloc[0]

def get_mode_if_significant(series, v_min, v_max, threshold=0.05):
    """
    Filters values between v_min and v_max, then returns the mode 
    if it represents more than 'threshold'% of the total data.
    """
    if series.empty:
        return None
    
    # 1. Keep only values in the business range (e.g., 1 to 4 for plan)
    in_range = series[series.between(v_min, v_max)]
    
    if in_range.empty:
        return series.mode().iloc[0] if not series.mode().empty else None

    # 2. Calculate frequencies on the total series (to keep the real proportion)
    counts = series.value_counts(normalize=True)
    
    # 3. Find the mode in our range
    mode_in_range = in_range.mode().iloc[0]
    
    # 4. If this mode is significant (> 5%), take it
    if counts.get(mode_in_range, 0) >= threshold:
        return mode_in_range
    
    # 5. Otherwise, return the mode (most frequent value)
    return series.mode().iloc[0]


Analyzes anomalies by characteristic, displaying them only if they
represent at least threshold_pct % of cases for the given route.

In [ ]:

def analyze_inconsistencies_by_feature(df, columns_to_check, threshold_pct=0):
    """
    Analyzes anomalies by feature, displaying them only if they 
    represent at least threshold_pct % of cases for the given road.
    """
    for col in columns_to_check:
        if col not in df.columns:
            continue
            
        print(f"\n{'='*60}")
        print(f"🔍 FEATURE ANALYSIS: {col} (Threshold: {threshold_pct}%)")
        print(f"{'='*60}")
        
        # Identify groups (com, adr) with multiple unique values
        inconsistencies = df.groupby(['com', 'adr'])[col].nunique()
        groups_with_issues = inconsistencies[inconsistencies > 1].index
        
        if len(groups_with_issues) == 0:
            print(f"✅ No anomaly detected for '{col}'.")
            continue

        for com, road in groups_with_issues:
            df_road = df[(df['com'] == com) & (df['adr'] == road)]
            total_events = len(df_road)
            
            # Calculate relative frequencies (%)
            counts = df_road[col].value_counts()
            percents = (counts / total_events) * 100
            
            # Determine the mode
            road_mode = counts.index[0]
            mode_count = counts.iloc[0]
            mode_pct = percents.iloc[0]
            
            # Look for anomalies that exceed the n% threshold
            # Ignore the first index (the mode)
            anomalies = counts.iloc[1:]
            significant_anomalies = anomalies[percents.iloc[1:] >= threshold_pct]

            # Display the road only if at least one anomaly is significant
            if not significant_anomalies.empty:
                print(f"\n🏙️  Municipality: {com} | 📍 Road: {road}")
                print(f"   [Mode]      {col} = {road_mode:<5} ({mode_count}/{total_events} occ. - {mode_pct:.1f}%)")
                
                for val, count in significant_anomalies.items():
                    val_pct = percents[val]
                    print(f"   ⚠️ [Anomaly] {col} = {val:<5} ({count}/{total_events} occ. - {val_pct:.1f}%)")

# Aggregation
agg_map = {
    'lat': get_mode, #OK
    'long': get_mode, #OK
    'dep': get_mode, #OK
    'com': get_mode, #OK
    'agg': get_mode, #OK even if some roads cross urban areas
    
    # Road identification
    'catr': get_mode, #OK
    'v1': get_mode, #OK
    
    # Physical characteristics & Risks
    'prof': get_mode,  #OK
    'plan': lambda s: get_max_if_significant(s, 1, 4), # most "dangerous" value if more than 5% and between 1 and 4
    'infra': lambda s: get_max_if_significant(s, 1, 7), # most "dangerous" value if more than 5% and between 1 and 7
    
    # Regulation & Dimensions
    'vma': get_mode, #OK
    'circ': get_mode,  #OK
    'vosp': lambda s: get_mode_if_significant(s, 1, 3), # most "dangerous" value if more than 5% and between 1 and 3
    'larrout': get_mode   #OK
}

print(agg_map.keys())

# List of technical columns to verify according to your schema
cols_to_check = agg_map.keys()

# Launch the analysis
analyze_inconsistencies_by_feature(df_secteur, cols_to_check,0.05)


In [ ]:


# Aggregation
agg_map = {
    'lat': get_mode,
    'long': get_mode,
    'dep': get_mode,
    'agg': get_mode,
    
    # Road identification
    'catr': get_mode,
    'v1': get_mode,
    
    # Physical characteristics & Risks
    'prof': get_mode,
    'plan': 'max', # most "dangerous" value
    'infra': 'max', #???
    
    # Regulation & Dimensions
    'vma': get_mode,
    'circ': get_mode,
    'vosp': get_mode,
    'larrout': get_mode
}


In [ ]:
# new columns for aggregation
agg_map_complement = {
    # Accident situation
    'situ': lambda s: get_mode_if_significant(s, 1, 6),  #OK
    
    # Intersection type (1:No intersection to 9:Square) 
    'int': lambda s: get_mode_if_significant(s, 2, 9,0.1), #OK
    
    # Collision type -> Significant mode
    'col': get_mode, #OK
    
    # Fixed obstacle hit (1:Vehicle to 17:Culvert) 
    'obs': lambda s: get_mode_if_significant(s, 1, 17), #OK
    
    # Mobile obstacle hit (1:Pedestrian to 6:Wild animal) 
    'obsm': lambda s: get_mode_if_significant(s, 1, 6), #OK
    
    # Pedestrian location -> Significant mode
    'locp': lambda s: get_mode_if_significant(s, 1, 8,0.1), #OK
    
}

# Usage
strut_car = df[["adr"]+accident_cols]
analyze_inconsistencies_by_feature(df_secteur, strut_car,5)


In [ ]:
def verify_schema_consistency(schema_mgr, expected_lists):
    """
    Verifies if the columns defined in the YAML schema match 
    the category lists defined in the code.
    """
    # 1. Get all schema columns
    all_schema_cols = schema_mgr.get_keys()
    
    # 2. Reverse the dictionary to group the schema by category
    # { 'TEMPORAL': ['jour', 'mois', ...], 'STRUCTURE': [...] }
    schema_by_category = {}
    for col in all_schema_cols:
        info = schema_mgr.schema['COLUMNS'][col]
        cat = info.get('category')
        if cat not in schema_by_category:
            schema_by_category[cat] = []
        schema_by_category[cat].append(col)

    # 3. Compare with your Python lists
    print(f"{'Category':<25} | {'Status':<10} | {'Details'}")
    print("-" * 70)

    for cat_name, expected_list in expected_lists.items():
        schema_list = schema_by_category.get(cat_name, [])
        
        # Columns present in your list but absent from YAML under this category
        missing_in_yaml = set(expected_list) - set(schema_list)
        
        # Columns in YAML but not in your Python list
        extra_in_yaml = set(schema_list) - set(expected_list)

        if not missing_in_yaml and not extra_in_yaml:
            print(f"{cat_name:<25} | ✅ OK")
        else:
            status = "⚠️ ERROR"
            print(f"{cat_name:<25} | {status}")
            if missing_in_yaml:
                print(f"   ❌ Missing in YAML (or wrong category): {missing_in_yaml}")
            if extra_in_yaml:
                print(f"   ➕ Extra in YAML: {extra_in_yaml}")




structure_cols = ["catr",
    "v1",
    "circ",
    "vosp",
    "prof",
    "plan",
    "larrout",
    "infra",
    "vma",
    "dep",
    "com",
    "agg",
    "lat",
    "long",
]

date_cols = ["jour","mois","an",
"hrmn",
"week_day",
"Hours",
]

meteo_cols = ["lum","atm",'surf']

accident_features = [
    "situ",
    "int",
    "col",
    "obs",
    "obsm",
    "locp",
]

accident_cols = [
    "Num_Acc",
    "nbv",
    "senc",
    'choc',
    "manv",
    "motor",
    "place",
    "catv_cluster",
    "sexe",
    "trajet",
    "actp",
    "etatp",
    "age",
]

target_cols = [
    "count_indemne",
    "count_blesse_leger",
    "count_blesse_hosp",
    "count_tue",
]

# Preparation of reference lists
lists_to_verify = {
    "STRUCTURE": structure_cols,
    "TEMPORAL": date_cols,
    "ENVIRONMENT": meteo_cols,
    "CRASH_TYPOLOGY": accident_features,
    "OTHERS": accident_cols,
    "TARGET": target_cols
}


# Launch the verification
verify_schema_consistency(schema_mgr, lists_to_verify)


Build Bassens structure CSV


In [ ]:
agg_map.update(agg_map_complement)
df_secteur_agg = df_secteur.groupby(['com', 'adr']).agg(agg_map)

In [ ]:
expected= set(structure_cols) | set(accident_features) 
res = set(df_secteur_agg.columns.tolist()) - expected
print(list(res))
res = expected - set(df_secteur_agg.columns.tolist())
print(list(res))

df_secteur_agg = df_secteur_agg.reset_index()
df_secteur_agg


In [ ]:
cols_ = [
    "locp_-1", "locp_0", "locp_1", "locp_2", "locp_3", "locp_4", "locp_5", "locp_6", "locp_7", "locp_8", "locp_9",
    "obs_0", "obs_1", "obs_2", "obs_3", "obs_4", "obs_5", "obs_6", "obs_7", "obs_8", "obs_9", "obs_10", "obs_11", "obs_12", "obs_13", "obs_14", "obs_15", "obs_16", "obs_17",
    "obsm_0", "obsm_1", "obsm_2", "obsm_4", "obsm_5", "obsm_6", "obsm_9"
]

df_secteur_agg = df_secteur_agg.copy()

for col_name in cols_:
    # On sépare le préfixe de la valeur (ex: 'locp' et '-1')
    # rsplit('_', 1) gère correctement le cas 'locp_-1'
    prefix, value = col_name.rsplit('_', 1)
    
    # On crée la colonne : 1 si la valeur correspond, sinon 0
    # On convertit 'value' en int pour la comparaison avec les données du DF
    df_secteur_agg[col_name] = (df_secteur_agg[prefix] == int(value)).astype(int)

df_secteur_agg.drop(columns=['locp', 'obs', 'obsm'], inplace=True)
df_secteur_agg.to_csv("../data/ref/road_secteur_bassens2.csv", sep=';')

In [ ]:
df_secteur_agg.to_csv("../data/ref/road_secteur_bassens_ref.csv", sep=';')
print(df_secteur_agg.columns)

In [ ]:
def describe_schema_dict(schema_dict):
    """
    Generates structured documentation from the schema dictionary.
    """
    # 1. Grouping by category
    custom_order = [
        'STRUCTURE', 
        'CRASH_TYPOLOGY', 
        'ENVIRONMENT', 
        'TEMPORAL', 
        'OTHERS', 
        'TARGET'
    ]
    categories = {}
    for col, info in schema_dict.items():
        cat = info.get('category', 'OTHERS')
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(col)

    # 3. Create ordered list of present categories
    # Take the defined order, and add unexpected categories at the end
    existing_cats = list(categories.keys())
    ordered_cats = [c for c in custom_order if c in existing_cats]
    remaining_cats = [c for c in existing_cats if c not in custom_order]
    final_category_list = ordered_cats + sorted(remaining_cats)

    # 2. Build the report
    lines = ["# 📘 DATA SCHEMA DOCUMENTATION", "="*40]
    
    for cat in final_category_list:
        lines.append(f"\n### 📂 {cat}")
        lines.append("-" * 30)
        
        for col in sorted(categories[cat]):
            info = schema_dict[col]
            desc = info.get('description', 'No description')
            dtype = info.get('type', 'unknown')
            
            # Badge to indicate if the column is used by the model
            fit_tag = " [Model Input]" if info.get('use_for_fit') else ""
            
            lines.append(f"**{col:<12}** | {dtype:<8} | {desc}{fit_tag}")
            
    return "\n".join(lines)

print(describe_schema_dict(schema_mgr.schema['COLUMNS']))
